# Week 6 – Integrative Capstone Project and Evaluation

This notebook demonstrates a complete Data Science pipeline using Python: data acquisition, cleaning/preprocessing, EDA, supervised learning, unsupervised learning, evaluation, and insights.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

np.random.seed(42)


## 1. Problem Definition

The capstone uses two public datasets to demonstrate complementary Data Science tasks. The Breast Cancer Wisconsin dataset is used for supervised binary classification, while the Iris dataset is used for unsupervised clustering. This makes it possible to demonstrate both predictive and descriptive modeling in one end-to-end workflow.

## 2. Data Acquisition and Inspection

In [ ]:
bc = load_breast_cancer()
X = bc.data
y = bc.target

print("Breast Cancer shape:", X.shape)
print("Target classes:", np.unique(y))
print("Feature names:", bc.feature_names[:10], "...")

iris = load_iris()
Xi = iris.data
yi = iris.target

print("Iris shape:", Xi.shape)
print("Iris classes:", np.unique(yi))


## 3. Data Cleaning and Preprocessing

The datasets are supplied as structured numerical datasets. We check dimensions and labels, use stratified splitting for the supervised task, and standardize features. For the supervised model, scaling is performed inside a Pipeline so the scaler is fitted only on training data, reducing data leakage risk.

In [ ]:
print("Breast Cancer missing values:", np.isnan(X).sum())
print("Iris missing values:", np.isnan(Xi).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)


## 4. Exploratory Data Analysis

In [ ]:
print(pd.Series(y).value_counts().sort_index())

summary = pd.DataFrame(X, columns=bc.feature_names).describe()
display(summary.head())


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(X[:, 0], bins=25)
plt.xlabel("First Breast Cancer Feature")
plt.ylabel("Frequency")
plt.title("Distribution of a Breast Cancer Feature")
plt.show()


## 5. Supervised Learning – Logistic Regression

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("Supervised Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.show()


## 6. Unsupervised Learning – K-Means

In [ ]:
scaler_i = StandardScaler()
Xi_s = scaler_i.fit_transform(Xi)

sil_scores = {}

for k in range(2, 7):
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = km_temp.fit_predict(Xi_s)
    sil_scores[k] = silhouette_score(Xi_s, labels_temp)

print("Silhouette scores:", sil_scores)

best_k = max(sil_scores, key=sil_scores.get)
print("Selected k:", best_k)


In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
iris_labels = km.fit_predict(Xi_s)

print("Final silhouette score:", round(silhouette_score(Xi_s, iris_labels), 4))


In [ ]:
plt.figure(figsize=(7,4))
plt.bar(list(sil_scores.keys()), list(sil_scores.values()))
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("K-Means Cluster Selection")
plt.show()


In [ ]:
pca = PCA(n_components=2)
Z = pca.fit_transform(Xi_s)

plt.figure(figsize=(7,5))
plt.scatter(Z[:,0], Z[:,1], c=iris_labels)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(f"Iris K-Means Clusters (k={best_k})")
plt.show()


## 7. Insights and Recommendations

- The supervised Logistic Regression model provides a strong baseline for binary classification.
- Scaling and a train/test split help create a reliable evaluation workflow.
- The confusion matrix shows where classification errors occur.
- K-Means reveals natural groupings in the Iris feature space without using labels during clustering.
- Silhouette analysis provides a quantitative basis for selecting the cluster count.
- Future work can compare multiple supervised models, tune hyperparameters, use cross-validation, and compare K-Means with hierarchical clustering.

## 8. Conclusion

This capstone integrates the major internship skills into one pipeline: acquisition, preprocessing, EDA, supervised modeling, unsupervised modeling, evaluation, visualization, and recommendations.